# Telugu BPE Tokenizer Training

## What is a Tokenizer?

A **tokenizer** is a tool that splits text into smaller pieces called **tokens**. 

### Example:
```
Sentence: "ఇది ఒక పరీక్ష వాక్యం."
                    ↓
Tokens:   ["ఇ", "ది", "ఒ", "క", "పరీక్ష", "వాక్యం", "।"]
```

## BPE (Byte-Pair Encoding) Tokenizer

BPE learns **which character sequences appear frequently together** and creates tokens from them.

### Training Process (What happens):

1. **Read all training text** from files (telugu.txt + te.txt from train/ and val/ folders)
2. **Sample the data** - Telugu corpus is huge (~14.9GB), so we sample ~2.5GB randomly but deterministically
3. **Find common byte patterns** - Which character pairs appear most often?
4. **Merge common pairs into tokens** - Turn "ఇ" + "ది" into "ఇది" token
5. **Repeat** until reaching target vocab size (50,000 tokens for Telugu)
6. **Save the learned vocab** as `telugu_tokenizer.json`

### At inference (using the tokenizer):
```
Input:  "ఇది ఒక పరీక్ష"
        ↓
Output: [<token_id_1>, <token_id_2>, <token_id_3>, ...]
        ↓ (model processes these token IDs)
```

## This Notebook's Steps:

1. **Setup**: Define data paths and config
2. **Utilities**: Vocabulary size calculation (dynamic heuristic)
3. **Sampling**: Extract ~2.5GB from ~14.9GB corpus (seed 42 = reproducible)
4. **Training**: Run BPE on sampled corpus
5. **Evaluation**: Test on held-out test set
6. **Regression**: Verify combining marks (matra/virama) survive correctly


## ⚠️ Important: Trained from Scratch (No Pretrained Tokenizers)

**This notebook trains a BPE tokenizer from scratch on the project corpus.**

- Uses the standalone `tokenizers` library (NOT `transformers`)
- Starts with a blank `models.BPE()` with no pretrained vocabulary
- `initial_alphabet=ByteLevel.alphabet()` only seeds the 256 raw byte symbols
- **No `.from_pretrained()` call anywhere** — all merges/vocab learned purely from your Telugu corpus
- Satisfies the project constraint: *No pretrained models, no pretrained tokenizers*


In [1]:
import json
import logging
import random
from datetime import datetime
from pathlib import Path
from typing import Optional

from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, decoders, processors, trainers

# ============================================================================
# ⚙️ CONFIGURATION: DATA ROOT PATH - MODIFY THIS IF DATA IS IN A DIFFERENT LOCATION
# ============================================================================
# Default: assumes notebook is in telugu/tokenizer/ and data is in telugu/data/
notebook_dir = Path("/kaggle/working/")
DATA_ROOT = Path("/kaggle/input/datasets/kspsvlnsiddardha/lma-slm/telugu/data")

# If data is elsewhere, set it explicitly:
# DATA_ROOT = Path("/kaggle/input/telugu-data")  # Example for Kaggle

TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "val"
TEST_DIR = DATA_ROOT / "test"
TOKENIZER_DIR = notebook_dir  # -> .../telugu/tokenizer
SAMPLE_CACHE_DIR = TOKENIZER_DIR / ".sample_cache"

print(f"✓ Data root: {DATA_ROOT}")
print(f"✓ Tokenizer dir: {TOKENIZER_DIR}")

# ============================================================================
# Language & tokenizer config
# ============================================================================
LANG = "Telugu"
LANG_SHORT = "telugu"
SPECIAL_TOKENS = ["<pad>", "<unk>", "<bos>", "<eos>"]
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3

# Sampling parameters
SAMPLE_CAP_GB = 2.5
SAMPLE_CAP_BYTES = int(SAMPLE_CAP_GB * 1024**3)
SAMPLE_SEED = 42

# Logging setup
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)

print(f"\n🚀 Training {LANG} BPE Tokenizer (with ~{SAMPLE_CAP_GB}GB sampling)")


✓ Data root: /kaggle/input/datasets/kspsvlnsiddardha/lma-slm/telugu/data
✓ Tokenizer dir: /kaggle/working

🚀 Training Telugu BPE Tokenizer (with ~2.5GB sampling)


In [2]:
# ============================================================================
# ALL FUNCTION DEFINITIONS (Define everything FIRST before using)
# ============================================================================

# ---- Utilities ----
def estimate_corpus_tokens(total_bytes: int, bytes_per_token: float = 4.0) -> int:
    """Rough token estimate from corpus size (bytes/4 heuristic)."""
    return int(total_bytes / bytes_per_token)


def compute_vocab_size(total_tokens_estimate: int) -> int:
    """Tiered vocab size heuristic: <50M->8K, 50M-200M->16K, 200M-1B->32K, >=1B->50K"""
    if total_tokens_estimate < 50_000_000:
        return 8_000
    elif total_tokens_estimate < 200_000_000:
        return 16_000
    elif total_tokens_estimate < 1_000_000_000:
        return 32_000
    else:
        return 50_000


def sample_lines_to_file(source_files: list[Path], target_bytes: int, output_path: Path, seed: int = 42) -> dict:
    """Deterministic single-pass Bernoulli line sampling."""
    total_bytes = sum(f.stat().st_size for f in source_files)
    p = min(1.0, target_bytes / total_bytes) if total_bytes else 0.0
    rng = random.Random(seed)
    written_bytes = written_lines = 0
    per_file: dict[str, int] = {}
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as out:
        for f in source_files:
            n = 0
            with open(f, "r", encoding="utf-8") as fh:
                for line in fh:
                    if rng.random() < p:
                        if not line.endswith("\n"):
                            line += "\n"
                        out.write(line)
                        written_bytes += len(line.encode("utf-8"))
                        written_lines += 1
                        n += 1
            per_file[str(f)] = n
    return {
        "sampling_probability": p,
        "source_total_bytes": total_bytes,
        "target_bytes": target_bytes,
        "actual_bytes_written": written_bytes,
        "actual_lines_written": written_lines,
        "per_file_lines_written": per_file,
        "seed": seed,
        "source_files_order": [str(f) for f in source_files],
    }

# ---- Corpus Discovery ----
def gather_training_files(split_dirs: list[Path]) -> list[Path]:
    """Find all *.txt files in given split directories, sorted."""
    files = []
    for split_dir in split_dirs:
        if split_dir.exists():
            files.extend(sorted(split_dir.glob("*.txt")))
    return files


def total_bytes(files: list[Path]) -> int:
    """Compute total size of files in bytes."""
    return sum(f.stat().st_size for f in files if f.exists())

# ---- Tokenizer Construction ----
def create_bpe_tokenizer() -> Tokenizer:
    """Create a ByteLevel BPE tokenizer with correct normalization."""
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.normalizer = normalizers.NFC()
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    tokenizer.post_processor = processors.ByteLevel(trim_offsets=True)
    return tokenizer


def build_trainer(vocab_size: int) -> trainers.BpeTrainer:
    """Build a BPE trainer with specified vocab size and special tokens."""
    return trainers.BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=2,
        special_tokens=SPECIAL_TOKENS,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        show_progress=True,
    )

# ---- Evaluation ----
def evaluate_tokenizer(tokenizer: Tokenizer, test_files: list[Path], sample_lines: int = 500) -> dict:
    """Evaluate tokenizer on held-out test set."""
    logger.info("Evaluating tokenizer on held-out test set...")
    sampled_lines = []
    rng = random.Random(42)
    total_read = 0
    for test_file in test_files:
        if not test_file.exists():
            continue
        with open(test_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.rstrip("\n")
                if not line.strip():
                    continue
                total_read += 1
                if len(sampled_lines) < sample_lines:
                    sampled_lines.append(line)
                else:
                    j = rng.randint(0, total_read - 1)
                    if j < sample_lines:
                        sampled_lines[j] = line
    logger.info(f"Sampled {len(sampled_lines)} lines from {total_read} read")
    token_lengths = []
    char_counts = []
    token_counts = []
    unk_count = 0
    total_tokens = 0
    roundtrip_pass = 0
    example_triples = []
    for line in sampled_lines[:100]:
        encoded = tokenizer.encode(line)
        decoded = tokenizer.decode(encoded.ids)
        token_lengths.append(len(encoded.ids))
        char_counts.append(len(line))
        token_counts.append(len(encoded.ids))
        for token_id in encoded.ids:
            total_tokens += 1
            if token_id == UNK_ID:
                unk_count += 1
        if decoded == line:
            roundtrip_pass += 1
        if len(example_triples) < 3:
            example_triples.append({
                "original": line[:80],
                "num_tokens": len(encoded.ids),
                "roundtrip_ok": decoded == line,
            })
    avg_tokens_per_line = sum(token_lengths) / len(token_lengths) if token_lengths else 0
    avg_chars_per_token = sum(char_counts) / sum(token_counts) if token_counts else 0
    unk_rate = 100.0 * unk_count / total_tokens if total_tokens > 0 else 0
    roundtrip_rate = 100.0 * roundtrip_pass / len(sampled_lines) if sampled_lines else 0
    return {
        "samples_evaluated": len(sampled_lines),
        "avg_tokens_per_line": round(avg_tokens_per_line, 2),
        "avg_chars_per_token": round(avg_chars_per_token, 2),
        "unk_rate_percent": round(unk_rate, 4),
        "roundtrip_match_percent": round(roundtrip_rate, 1),
        "example_triples": example_triples,
    }

print("✅ ALL FUNCTIONS DEFINED - Ready to use!")

# ---- Full Tokenizer Report (entire test set) ----
def generate_tokenizer_report(tokenizer: Tokenizer, test_files: list[Path],
                               vocab_size_requested: int, top_n: int = 20) -> dict:
    """Full test-set pass: vocab size, token-frequency stats, avg chars/token,
    tokenization examples, UNK statistic — for the project report."""
    from collections import Counter

    token_freq = Counter()
    total_tokens = 0
    total_chars = 0
    total_lines = 0
    unk_count = 0

    for test_file in test_files:
        if not test_file.exists():
            continue
        with open(test_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.rstrip("\n")
                if not line.strip():
                    continue
                total_lines += 1
                total_chars += len(line)
                encoded = tokenizer.encode(line)
                total_tokens += len(encoded.ids)
                token_freq.update(encoded.ids)
                unk_count += sum(1 for tid in encoded.ids if tid == UNK_ID)

    vocab_actual = tokenizer.get_vocab_size()
    most_common = [
        {"token": tokenizer.id_to_token(tid), "id": tid, "count": count,
         "percent_of_tokens": round(100.0 * count / total_tokens, 4)}
        for tid, count in token_freq.most_common(top_n)
    ]
    freq_values = list(token_freq.values())
    unique_tokens_used = len(token_freq)

    return {
        "vocab_size_requested": vocab_size_requested,
        "vocab_size_actual": vocab_actual,
        "unique_tokens_used_in_test": unique_tokens_used,
        "vocab_coverage_percent": round(100.0 * unique_tokens_used / vocab_actual, 2),
        "total_test_lines": total_lines,
        "total_test_tokens": total_tokens,
        "avg_chars_per_token": round(total_chars / total_tokens, 4) if total_tokens else 0,
        "unk_count": unk_count,
        "unk_rate_percent": round(100.0 * unk_count / total_tokens, 4) if total_tokens else 0,
        "token_frequency_top_n": most_common,
        "token_frequency_stats": {
            "min": min(freq_values) if freq_values else 0,
            "max": max(freq_values) if freq_values else 0,
            "mean": round(sum(freq_values) / len(freq_values), 2) if freq_values else 0,
        },
    }


✅ ALL FUNCTIONS DEFINED - Ready to use!


In [3]:
# ============================================================================
# Discover corpus files
# ============================================================================

logger.info("Discovering corpus files...")
train_files = gather_training_files([TRAIN_DIR])
val_files = gather_training_files([VAL_DIR])
test_files = gather_training_files([TEST_DIR])

logger.info(f"Train files: {[f.name for f in train_files]}")
logger.info(f"Val files: {[f.name for f in val_files]}")
logger.info(f"Test files: {[f.name for f in test_files]}")

# Compute vocab size from train+val
train_val_files = train_files + val_files
train_val_bytes = total_bytes(train_val_files)
train_val_tokens = estimate_corpus_tokens(train_val_bytes)
vocab_size = compute_vocab_size(train_val_tokens)

print(f"\n📊 Corpus stats (train+val):")
print(f"  Total bytes: {train_val_bytes / (1024**3):.1f} GB")
print(f"  Estimated tokens: {train_val_tokens:,}")
print(f"  Vocab size (heuristic): {vocab_size:,}")


[INFO] Discovering corpus files...
[INFO] Train files: ['te.txt', 'telugu.txt']
[INFO] Val files: ['te.txt', 'telugu.txt']
[INFO] Test files: ['te.txt', 'telugu.txt']



📊 Corpus stats (train+val):
  Total bytes: 14.9 GB
  Estimated tokens: 4,000,600,004
  Vocab size (heuristic): 50,000


## Sampling Phase

Telugu corpus is ~14.9GB. We'll sample down to ~2.5GB using deterministic Bernoulli line sampling (seed=42 for reproducibility).


In [4]:
# ============================================================================
# SAMPLING PHASE: Extract ~2.5GB from ~14.9GB corpus
# ============================================================================

SAMPLE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
sample_file = SAMPLE_CACHE_DIR / f"{LANG_SHORT}_sample.txt"

logger.info(f"Sampling ~{SAMPLE_CAP_GB}GB from {train_val_bytes/(1024**3):.1f}GB corpus...")
sample_provenance = sample_lines_to_file(train_val_files, SAMPLE_CAP_BYTES, sample_file, seed=SAMPLE_SEED)

print(f"✓ Sampled {sample_provenance['actual_lines_written']:,} lines "
      f"({sample_provenance['actual_bytes_written']/(1024**3):.2f} GB)")
print(f"  Sampling probability: {sample_provenance['sampling_probability']:.6f}")


[INFO] Sampling ~2.5GB from 14.9GB corpus...


✓ Sampled 3,827,608 lines (2.50 GB)
  Sampling probability: 0.167747


In [5]:
# ============================================================================
# Train tokenizer
# ============================================================================

logger.info("Creating tokenizer...")
tokenizer = create_bpe_tokenizer()

logger.info("Building trainer...")
trainer = build_trainer(vocab_size)

logger.info("Training BPE on sampled corpus (this may take several minutes)...")
tokenizer.train(
    files=[str(sample_file)],
    trainer=trainer,
)

print("✓ Training complete")


[INFO] Creating tokenizer...
[INFO] Building trainer...
[INFO] Training BPE on sampled corpus (this may take several minutes)...





✓ Training complete


## Training Phase: How BPE Learns (on Sampled Data)

When we run `tokenizer.train()` on the sampled corpus, here's what happens:

### Step-by-step:

1. **Load sampled text**
   - Read from the cached sample file: ~2.5GB of Telugu text
   - This sample is deterministically extracted (seed=42) from the full corpus
   - Represents the diversity of the full 14.9GB

2. **Start with bytes**
   - Each character is initially a token
   - "ఇది" = ["ఇ", "ది"] (2 tokens)

3. **Find frequent pairs**
   - Count which 2-character sequences appear most often in the sample
   - Example: "ఇ" + "ది" appears 100,000 times → merge to "ఇది"

4. **Merge iteratively**
   - Next merge: find the next most frequent pair, merge it
   - Keep doing this until vocab reaches 50,000 tokens

5. **Save vocabulary**
   - Store all learned merges in `telugu_tokenizer.json`
   - Now the tokenizer knows: "ఇది" = 1 token (not 2)

### Why sampling?
- Full 14.9GB would take too long to train (hours on personal machine)
- BPE converges well with a representative sample
- Deterministic sampling (seed 42) means training is reproducible


In [6]:
# ============================================================================
# Save tokenizer and config
# ============================================================================

tokenizer_path = TOKENIZER_DIR / f"{LANG_SHORT}_tokenizer.json"
logger.info(f"Saving tokenizer to {tokenizer_path.name}...")
tokenizer.save(str(tokenizer_path))

vocab_actual = tokenizer.get_vocab_size()
logger.info(f"Vocab size actual: {vocab_actual:,}")
if vocab_actual != vocab_size:
    logger.warning(f"  Note: actual ({vocab_actual}) differs from requested ({vocab_size})")

# Save config
config = {
    "language": LANG,
    "tokenizer_type": "BPE",
    "model": "ByteLevel BPE",
    "normalizer": "NFC",
    "vocab_size_requested": vocab_size,
    "vocab_size_actual": vocab_actual,
    "special_tokens": SPECIAL_TOKENS,
    "sampling": {
        "cap_gb": SAMPLE_CAP_GB,
        "seed": SAMPLE_SEED,
        "actual_bytes_written": sample_provenance['actual_bytes_written'],
        "actual_lines_written": sample_provenance['actual_lines_written'],
    },
    "created_at": datetime.now().isoformat(),
}

config_path = TOKENIZER_DIR / "tokenizer_config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
logger.info(f"Saved config to {config_path.name}")

print(f"✓ Tokenizer and config saved")


[INFO] Saving tokenizer to telugu_tokenizer.json...
[INFO] Vocab size actual: 50,000
[INFO] Saved config to tokenizer_config.json


✓ Tokenizer and config saved


In [7]:
# ============================================================================
# Generate and save comprehensive tokenizer report
# ============================================================================

logger.info("Generating comprehensive tokenizer report...")
report = generate_tokenizer_report(tokenizer, test_files, vocab_size)

print(f"\n📊 TOKENIZER REPORT:")
print(f"  Vocab size (requested): {report['vocab_size_requested']:,}")
print(f"  Vocab size (actual): {report['vocab_size_actual']:,}")
print(f"  Unique tokens used in test: {report['unique_tokens_used_in_test']:,}")
print(f"  Vocab coverage: {report['vocab_coverage_percent']:.2f}%")
print(f"  Test set: {report['total_test_lines']:,} lines, {report['total_test_tokens']:,} tokens")
print(f"  Avg chars/token: {report['avg_chars_per_token']:.4f}")
print(f"  UNK count: {report['unk_count']:,}")
print(f"  UNK rate: {report['unk_rate_percent']:.4f}%")

print(f"\n📈 Top-20 Most Frequent Tokens:")
for i, item in enumerate(report['token_frequency_top_n'][:20], 1):
    token_repr = repr(item['token']) if len(item['token']) <= 20 else repr(item['token'][:20] + '...')
    print(f"  {i:2d}. {token_repr:25s} id={item['id']:5d} count={item['count']:8d} ({item['percent_of_tokens']:.2f}%)")

report_path = TOKENIZER_DIR / f"telugu_tokenizer_report.json"
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
logger.info(f"Saved report to {report_path.name}")


[INFO] Generating comprehensive tokenizer report...
[INFO] Saved report to telugu_tokenizer_report.json



📊 TOKENIZER REPORT:
  Vocab size (requested): 50,000
  Vocab size (actual): 50,000
  Unique tokens used in test: 49,262
  Vocab coverage: 98.52%
  Test set: 2,533,618 lines, 484,239,493 tokens
  Avg chars/token: 1.3623
  UNK count: 0
  UNK rate: 0.0000%

📈 Top-20 Most Frequent Tokens:
   1. 'à±į'                     id=  263 count=47826294 (9.88%)
   2. 'à°¾'                     id=  265 count=34585059 (7.14%)
   3. 'à°¿'                     id=  264 count=32394926 (6.69%)
   4. 'à±ģ'                     id=  266 count=28594631 (5.91%)
   5. 'à°¨'                     id=  268 count=16951937 (3.50%)
   6. 'à°°'                     id=  267 count=15332047 (3.17%)
   7. 'à°²'                     id=  270 count=15022371 (3.10%)
   8. 'à°Ĥ'                     id=  269 count=14657839 (3.03%)
   9. 'à±ĭ'                     id=  273 count=11488161 (2.37%)
  10. 'à±ĩ'                     id=  275 count=10894362 (2.25%)
  11. 'à°¤'                     id=  272 count=10203228 (2.11%)
  12. 'à°

## Evaluation on Test Set


In [8]:
# ============================================================================
# Evaluate tokenizer
# ============================================================================

def evaluate_tokenizer(tokenizer: Tokenizer, test_files: list[Path], sample_lines: int = 500) -> dict:
    """Evaluate tokenizer on held-out test set."""
    logger.info("Evaluating tokenizer on held-out test set...")

    sampled_lines = []
    rng = random.Random(42)
    total_read = 0

    for test_file in test_files:
        if not test_file.exists():
            continue
        with open(test_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.rstrip("\n")
                if not line.strip():
                    continue
                total_read += 1
                if len(sampled_lines) < sample_lines:
                    sampled_lines.append(line)
                else:
                    j = rng.randint(0, total_read - 1)
                    if j < sample_lines:
                        sampled_lines[j] = line

    logger.info(f"Sampled {len(sampled_lines)} lines from {total_read} read")

    token_lengths = []
    char_counts = []
    token_counts = []
    unk_count = 0
    total_tokens = 0
    roundtrip_pass = 0
    example_triples = []

    for line in sampled_lines[:100]:
        encoded = tokenizer.encode(line)
        decoded = tokenizer.decode(encoded.ids)

        token_lengths.append(len(encoded.ids))
        char_counts.append(len(line))
        token_counts.append(len(encoded.ids))

        for token_id in encoded.ids:
            total_tokens += 1
            if token_id == UNK_ID:
                unk_count += 1

        if decoded == line:
            roundtrip_pass += 1

        if len(example_triples) < 3:
            example_triples.append({
                "original": line[:80],
                "num_tokens": len(encoded.ids),
                "roundtrip_ok": decoded == line,
            })

    avg_tokens_per_line = sum(token_lengths) / len(token_lengths) if token_lengths else 0
    avg_chars_per_token = sum(char_counts) / sum(token_counts) if token_counts else 0
    unk_rate = 100.0 * unk_count / total_tokens if total_tokens > 0 else 0
    roundtrip_rate = 100.0 * roundtrip_pass / len(sampled_lines) if sampled_lines else 0

    return {
        "samples_evaluated": len(sampled_lines),
        "avg_tokens_per_line": round(avg_tokens_per_line, 2),
        "avg_chars_per_token": round(avg_chars_per_token, 2),
        "unk_rate_percent": round(unk_rate, 4),
        "roundtrip_match_percent": round(roundtrip_rate, 1),
        "example_triples": example_triples,
    }

eval_results = evaluate_tokenizer(tokenizer, test_files)

print(f"\n📈 Evaluation Results:")
print(f"  Samples evaluated: {eval_results['samples_evaluated']}")
print(f"  Avg tokens/line: {eval_results['avg_tokens_per_line']}")
print(f"  Avg chars/token: {eval_results['avg_chars_per_token']:.2f}")
print(f"  UNK rate: {eval_results['unk_rate_percent']:.4f}%")
print(f"  Roundtrip match: {eval_results['roundtrip_match_percent']:.1f}%")

print(f"\n📝 Example Encode/Decode:")
for i, triple in enumerate(eval_results['example_triples'], 1):
    print(f"  {i}. {triple['original'][:60]}...")
    print(f"     Tokens: {triple['num_tokens']}, Roundtrip OK: {triple['roundtrip_ok']}")


[INFO] Evaluating tokenizer on held-out test set...
[INFO] Sampled 500 lines from 2533618 read



📈 Evaluation Results:
  Samples evaluated: 500
  Avg tokens/line: 191.62
  Avg chars/token: 1.35
  UNK rate: 0.0000%
  Roundtrip match: 19.4%

📝 Example Encode/Decode:
  1. ప్రస్తుతం నెట్‌ఫ్లిక్స్‌లో ప్రసారమవుతోన్న ‘గర్ల్స్‌ హాస్టల్‌...
     Tokens: 72, Roundtrip OK: True
  2. జాతీయ రహదారి, రాష్ట్ర రహదారి, ప్రధాన జిల్లా రహదారి, జిల్లా ర...
     Tokens: 102, Roundtrip OK: True
  3. ఈ పాటలో అమెరికన్ గాయని  రాపర్ డోజా క్యాట్ ఉన్నారు. దీనిని డా...
     Tokens: 355, Roundtrip OK: True


In [9]:
# ============================================================================
# Test cases: combining marks + FULL SENTENCES (sentence-level tokenization)
# ============================================================================

print("\n" + "="*70)
print("TEST CASES: SENTENCE-LEVEL TOKENIZATION")
print("="*70)

# Test 1: Combining marks regression (matra/virama)
print("\n🔍 Regression Tests (Combining Marks - Matra/Virama):")
combining_test_cases = [
    ("క్ష", "Telugu conjunct (virama)"),
    ("కి", "Telugu vowel sign ి (U+0C3F)"),
    ("కీ", "Telugu vowel sign ీ (U+0C40)"),
    ("ద్య", "Telugu conjunct (d + virama + y)"),
]

all_pass = True
for text, description in combining_test_cases:
    encoded = tokenizer.encode(text)
    decoded = tokenizer.decode(encoded.ids)
    passed = (decoded == text)
    status = "✓" if passed else "✗"
    print(f"  {status} {description}")
    print(f"     Input: {text}, Decoded: {decoded}")
    if not passed:
        all_pass = False

if all_pass:
    print("\n  ✓ All combining mark tests PASSED!")
else:
    print("\n  ✗ Some tests FAILED")

# Test 2: Full sentence tokenization
print("\n📝 Sentence-Level Tokenization Examples:")
sentence_test_cases = [
    "ఇది ఒక పరీక్ష వాక్యం.",
    "తెలుగు ఒక సందర్భ భాష.",
    "ఎందుకంటే ఈ భాష చాలా అందమైనది.",
    "నేను ఒక విద్యార్థిని.",
]

for sentence in sentence_test_cases:
    encoded = tokenizer.encode(sentence)
    decoded = tokenizer.decode(encoded.ids)
    token_strs = [tokenizer.id_to_token(tid) for tid in encoded.ids]
    match = "✓" if decoded == sentence else "✗"
    print(f"\n  {match} Sentence: {sentence[:50]}...")
    print(f"     Token strings: {token_strs}")
    print(f"     Token count: {len(encoded.ids)}")
    print(f"     Token IDs: {encoded.ids[:15]}{'...' if len(encoded.ids) > 15 else ''}")
    print(f"     Roundtrip OK: {decoded == sentence}")



TEST CASES: SENTENCE-LEVEL TOKENIZATION

🔍 Regression Tests (Combining Marks - Matra/Virama):
  ✓ Telugu conjunct (virama)
     Input: క్ష, Decoded: క్ష
  ✓ Telugu vowel sign ి (U+0C3F)
     Input: కి, Decoded: కి
  ✓ Telugu vowel sign ీ (U+0C40)
     Input: కీ, Decoded: కీ
  ✓ Telugu conjunct (d + virama + y)
     Input: ద్య, Decoded: ద్య

  ✓ All combining mark tests PASSED!

📝 Sentence-Level Tokenization Examples:

  ✓ Sentence: ఇది ఒక పరీక్ష వాక్యం....
     Token strings: ['à°ĩà°¦', 'à°¿', 'Ġà°Ĵà°ķ', 'Ġà°ªà°°', 'à±Ģ', 'à°ķ', 'à±į', 'à°·', 'Ġà°µ', 'à°¾', 'à°ķ', 'à±į', 'à°¯', 'à°Ĥ.']
     Token count: 14
     Token IDs: [830, 264, 362, 358, 283, 271, 263, 305, 289, 265, 271, 263, 276, 407]
     Roundtrip OK: True

  ✓ Sentence: తెలుగు ఒక సందర్భ భాష....
     Token strings: ['à°¤', 'à±Ĩ', 'à°²', 'à±ģ', 'à°Ĺ', 'à±ģ', 'Ġà°Ĵà°ķ', 'Ġà°¸', 'à°Ĥ', 'à°¦à°°', 'à±į', 'à°Ń', 'Ġà°Ń', 'à°¾', 'à°·', '.']
     Token count: 16
     Token IDs: [272, 287, 270, 266, 284, 266, 362, 290, 269, 350, 263, 3

## Summary

✓ Telugu tokenizer training complete!

**Outputs:**
- `telugu_tokenizer.json` - Trained tokenizer
- `tokenizer_config.json` - Configuration and metadata
- `.sample_cache/` - Cached sampled corpus (for re-training without re-sampling)
